In [ ]:
# !pip install umap-learn

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import umap.umap_ as umap
import warnings

# Set plotting style for publication-quality figures
sns.set(style='whitegrid', context='talk', font_scale=1.2)

def compute_embeddings(features, model_name='pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb'):
    """
    Compute embeddings for a list of feature names using a pretrained SentenceTransformer model.

    Args:
        features (list of str): List of feature names.
        model_name (str): Name of the pretrained SentenceTransformer model.

    Returns:
        np.ndarray: Array of embeddings.
    """
    model = SentenceTransformer(model_name)
    embeddings = model.encode(features, show_progress_bar=True)
    return embeddings

def determine_optimal_clusters(embeddings, max_clusters=30,avoid_singletone_clusters=False):
    """
    Determine the optimal number of clusters using the Silhouette Score.

    Args:
        embeddings (np.ndarray): Array of embeddings.
        max_clusters (int): Maximum number of clusters to try.

    Returns:
        int: Optimal number of clusters.
        list: List of valid cluster numbers.
        list: Silhouette scores for each valid number of clusters.
    """
    warnings.filterwarnings("ignore")
    silhouette_scores = []
    valid_n_clusters = []
    cluster_range = range(2, max_clusters+1)
    for n_clusters in cluster_range:
        try:
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=15)
            labels = kmeans.fit_predict(embeddings)
            # Check if any cluster has only one sample
            unique, counts = np.unique(labels, return_counts=True)
            if avoid_singletone_clusters:
                if np.any(counts < 2):
                    print(f"Skipping n_clusters = {n_clusters}: at least one cluster has fewer than 2 samples.")
                    continue
            score = silhouette_score(embeddings, labels)
            silhouette_scores.append(score)
            valid_n_clusters.append(n_clusters)
            print(f"n_clusters = {n_clusters}, silhouette score = {score:.4f}")
        except Exception as e:
            print(f"Exception for n_clusters = {n_clusters}: {e}")
            continue
    if len(silhouette_scores) == 0:
        raise ValueError("No valid silhouette scores were computed.")
    optimal_n_clusters = valid_n_clusters[np.argmax(silhouette_scores)]
    return optimal_n_clusters, valid_n_clusters, silhouette_scores

def cluster_embeddings(embeddings, n_clusters):
    """
    Cluster embeddings using KMeans.

    Args:
        embeddings (np.ndarray): Array of embeddings.
        n_clusters (int): Number of clusters.

    Returns:
        np.ndarray: Cluster labels for each embedding.
    """
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(embeddings)
    return cluster_labels

def assign_features_to_clusters(features, cluster_labels):
    """
    Assign features to their corresponding clusters.

    Args:
        features (list of str): List of feature names.
        cluster_labels (np.ndarray): Cluster labels for each feature.

    Returns:
        pd.DataFrame: DataFrame with features and their assigned clusters.
    """
    df_clusters = pd.DataFrame({'feature': features, 'cluster': cluster_labels})
    return df_clusters

def visualize_clusters(embeddings, cluster_labels, method='umap', savefig=False, filename='clusters_visualization.png'):
    """
    Visualize clusters using dimensionality reduction techniques.

    Args:
        embeddings (np.ndarray): Array of embeddings.
        cluster_labels (np.ndarray): Cluster labels for each embedding.
        method (str): Dimensionality reduction method ('umap' or 'tsne').
        savefig (bool): Whether to save the figure.
        filename (str): Filename for saving the figure.

    Returns:
        None
    """
    if method == 'umap':
        reducer = umap.UMAP(n_neighbors=2, # 15
                            min_dist=0.05#0.1,
                            ,metric='cosine', random_state=42)
        embeddings_2d = reducer.fit_transform(embeddings)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
        embeddings_2d = reducer.fit_transform(embeddings)
    else:
        raise ValueError("Method must be 'umap' or 'tsne'.")

    n_clusters = len(np.unique(cluster_labels))
    palette = sns.color_palette("hls", n_clusters)

    df_vis = pd.DataFrame({
        'Component 1': embeddings_2d[:,0],
        'Component 2': embeddings_2d[:,1],
        'Cluster': cluster_labels
    })

    plt.figure(figsize=(12,8))
    sns.scatterplot(
        x='Component 1', y='Component 2',
        hue='Cluster',
        palette=palette,
        data=df_vis,
        legend='full',
        s=100,
        alpha=0.8,
        edgecolor='k'
    )
    plt.title('Feature Clusters Visualization', fontsize=18)
    plt.xlabel('Component 1', fontsize=14)
    plt.ylabel('Component 2', fontsize=14)
    plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize=12)
    plt.tight_layout()
    if savefig:
        plt.savefig(filename, dpi=300, bbox_inches='tight', format='png')
    plt.show()

def main():
    """
    Main function to perform clustering on features and visualize the results.

    Returns:
        None
    """
    # List of features (as provided)
    features = [
        'SHBG',
        'Home area population density - urban or rural Scotland - Large Urban Area',
        'Apolipoprotein B to Apolipoprotein A1 ratio',
        '(AF) atrial fibrillation genetic risk',
        'Concentration of Medium HDL Particles',
        'Pulse wave reflection index',
        'Sleep - Overall average',
        'medication entry unable',
        'Bipolar and major depression status',
        'HDL cholesterol Blood biochemistry',
        'Oily fish intake Never',
        'Microalbumin in urine',
        'Triglycerides Blood biochemistry',
        'Total protein aliquot',
        'Neuroticism score',
        'Potassium in urine',
        'Urinary tract infection',
        'Sodium in urine',
        '(AAM) age at menopause genetic risk',
        'Health satisfaction',
        'Variation in diet Never/rarely',
        'Had other major operations Yes - you will be asked about this later by an interviewer',
        '(AST) asthma genetic risk',
        'Apolipoprotein A1',
        'Mental health conditions ever diagnosed by a professional of',
        'Apolipoprotein B Blood biochemistry',
        'Hot drink temperature Hot',
        'Direct bilirubin',
        'Concentration of Small HDL Particles',
        '(MEL) melanoma genetic risk',
        'HBA1C DF glycated haemoglobin genetic risk',
        'Average Diameter for LDL Particles',
        'POAG primary open angle glaucoma genetic risk',
        'Time spent outdoors in winter',
        'IOP intraocular pressure genetic risk',
        'Hearing difficulty problems',
        '(AMD) age-related macular degeneration genetic risk',
        'PC prostate cancer genetic risk',
        'Hip circumference',
        'Fluid intelligence score',
        'Concentration of IDL Particles',
        'Skin colour Dark olive',
        'Lipoprotein A Blood biochemistry',
        'Myopia diagnosis non myopic',
        'Home area population density  urban or rural Scotland  Accessible Rural',
        'Concentration of Very Small VLDL Particles',
        'Concentration of Small VLDL Particles',
        'Concentration of Medium VLDL Particles',
        'Complications of transplants and reattached limbs',
        'Cholesterol in IDL',
        'Frequency of depressed mood in last 2 weeks Not at all',
        'Concentration of HDL Particles',
        'Syncope and collapse',
        'Pleurisy pleural effusion',
        'Benign neoplasm of other parts of digestive system',
        'Home area population density  urban or rural England Wales  Urban  less sparse',
        'Cochlear implant',
        '(HDL) high density lipoprotein cholesterol genetic risk',
        'EBMDT estimated bone mineral density t score genetic risk',
        'medication simvastatin',
        'Started insulin within one year diagnosis of diabetes',
        'medication amlodipine',
        'BC breast cancer genetic risk',
        'Urinary incontinence',
        'emphysema chronic',
        'PSO psoriasis genetic risk',
        'Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor None of the above',
        'PD parkinson s disease genetic risk',
        'Tea intake',
        'Coffee intake',
        'Oily fish intake 2-4 times a week',
        'Arm fat percentage',
        'Apolipoprotein B',
        'enlarged prostate',
        'Urate',
        'Acetoacetate',
        'Contra-indications for spirometry',
        'Standing height',
        '3-Hydroxybutyrate',
        'Cholesterol in Very Small VLDL',
        'Age cataract diagnosed',
        'medication alendronate sodium',
        'Acetone',
        'Testosterone',
        'Z90.4 - Acquired absence of other parts of digestive tract',
        'Falls in the last year Only one fall',
        'Medication for cholesterol blood pressure or diabetes Medication Cholesterol lowering medication Blood pressure medication',
        'Hypercholesterolemia',
        '(LDL SF) low density lipoprotein cholesterol genetic risk',
        'high cholesterol',
        'Alcohol intake frequency Three or four times a week',
        'Ever addicted to any substance or behaviour',
        'Cataract',
        'Bipolar and major depression status No Bipolar or Depression',
        'Age heart attack diagnosed',
        'Water intake',
        'Urea',
        'Alanine',
        'Townsend deprivation index at recruitment',
        'Arthropathy NOS',
        'Weight (p21002)',
        'Never eat eggs dairy wheat sugar Wheat products Sugar or foods drinks containing sugar',
        'Never eat eggs, dairy, wheat, sugar Wheat products',
        'dietary changes in the last 5 years Yes, because of illness',
        'Overall health rating Fair',
        'Number of self reported non cancer illnesses',
        'Fractured/broken bones in last 5 years',
        '(CRC) bowel cancer genetic risk',
        'EOC epithelial ovarian cancer genetic risk',
        'Ever had bowel cancer screening',
        'VTE venous thromboembolic disease genetic risk',
        'Variation in diet Sometimes',
        'Cholesterol Blood biochemistry',
        'dietary changes in the last 5 years',
        'Number of treatments medications taken',
        'Other serious medical condition/disability diagnosed by doctor Yes - you will be asked about this later by an interviewer',
        'Z03.8 - Observation for other suspected diseases and conditions',
        'Non-cancer illness code, self-reported |',
        '(BMI) Body mass index (p21001)',
        'Long-standing illness, disability or infirmity',
        'Skin colour Fair',
        'medication ramipril',
        'Falls in the last year No falls',
        'medication chewable',
        'Never eat eggs, dairy, wheat, sugar Sugar or foods/drinks containing sugar',
        'Mouth/teeth dental problems None of the above',
        'Noninfectious gastroenteritis',
        'Frequency of tiredness / lethargy in last 2 weeks Not at all',
        'Never eat eggs, dairy, wheat, sugar I eat all of the above',
        'dietary changes in the last 5 years Yes, because of other reasons',
        'Chemotherapy',
        'Mineral and other dietary supplements None of the above',
        'Weight change compared with 1 year ago Yes - gained weight',
        '(BMI) body mass index genetic risk'
    ]
    # Create DataFrame of features
    df_features = pd.DataFrame({'feature': features})

    # Compute embeddings
    embeddings = compute_embeddings(df_features['feature'].tolist())

    # Determine optimal number of clusters
    optimal_n_clusters, valid_n_clusters, silhouette_scores = determine_optimal_clusters(embeddings, max_clusters=40)
    print(f"\nOptimal number of clusters: {optimal_n_clusters}")

    # Plot Silhouette Scores
    plt.figure(figsize=(8, 6))
    plt.plot(valid_n_clusters, silhouette_scores, marker='o')
    plt.title('Silhouette Score for Different Number of Clusters')
    plt.xlabel('Number of clusters')
    plt.ylabel('Silhouette Score')
    plt.xticks(valid_n_clusters)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Cluster embeddings with the optimal number of clusters
    cluster_labels = cluster_embeddings(embeddings, optimal_n_clusters)

    # Assign features to clusters
    df_clusters = assign_features_to_clusters(df_features['feature'].tolist(), cluster_labels)

    # Show number of features per cluster
    cluster_counts = df_clusters['cluster'].value_counts().sort_index()
    print("\nNumber of features per cluster:")
    print(cluster_counts)

    # Show features per cluster
    print("\nFeatures per cluster:")
    for cluster_num in sorted(df_clusters['cluster'].unique()):
        features_in_cluster = df_clusters[df_clusters['cluster'] == cluster_num]['feature'].tolist()
        print(f"\nCluster {cluster_num} (n={len(features_in_cluster)}):")
        for feature in features_in_cluster:
            print(f"- {feature}")

    # Optionally, output dataframe with features and their cluster
    # df_clusters.to_csv('features_clusters.csv', index=False)

    # Visualize clusters
    visualize_clusters(embeddings, cluster_labels, method='umap', savefig=False, filename='clusters_visualization.png')


In [ ]:
# !pip install 'bertopic[gensim,spacy,visualization]'
# !pip install umap-learn

## GPT Clusters
* Used O1 to define semantic clusters. then had it label each feature (I used the list of all features (feature_name, not raw) from all 8 tasks selected as interesting|novel + inlcuded their # counts across the 8 tasks).

In [ ]:
import plotly.express as px

###############################################################################
# 1) Create the master list of dictionaries: each row has:
#    {
#      "cluster": <one_of_the_11_clusters>,
#      "feature": <the_exact_feature_name_string>,
#      "count": <integer_count>
#    }
#
#   Assign each feature to a cluster based on its semantics.
###############################################################################

data = [
    # --- Genetic Risk (Polygenic Risk Scores) ---
    {"cluster": "Genetic Risk", "feature": "(AAM) age at menopause genetic risk", "count": 7},
    {"cluster": "Genetic Risk", "feature": "(AD) alzheimer's disease genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(AF) atrial fibrillation genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "(AMD) age-related macular degeneration genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "(AST) asthma genetic risk", "count": 5},
    {"cluster": "Genetic Risk", "feature": "(BMI) body mass index genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(CD) crohn's disease genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(CED) coeliac disease genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(CRC) bowel cancer genetic risk", "count": 2},
    {"cluster": "Genetic Risk", "feature": "(HDL) high density lipoprotein cholesterol genetic risk", "count": 4},
    {"cluster": "Genetic Risk", "feature": "(HT) hypertension genetic risk", "count": 2},
    {"cluster": "Genetic Risk", "feature": "(ISS) ischaemic stroke genetic risk", "count": 5},
    {"cluster": "Genetic Risk", "feature": "(LDL SF) low density lipoprotein cholesterol genetic risk", "count": 6},
    {"cluster": "Genetic Risk", "feature": "(MEL) melanoma genetic risk", "count": 8},
    {"cluster": "Genetic Risk", "feature": "(MS) multiple sclerosis genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(PC) prostate cancer genetic risk", "count": 4},
    {"cluster": "Genetic Risk", "feature": "(RA) rheumatoid arthritis genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(SLE) systemic lupus erythematosus genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(T2D) type 2 diabetes genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "(UC) ulcerative colitis genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "AD alzheimer s disease genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "AMD age related macular degeneration genetic risk", "count": 2},
    {"cluster": "Genetic Risk", "feature": "BC breast cancer genetic risk", "count": 5},
    {"cluster": "Genetic Risk", "feature": "BD bipolar disorder genetic risk", "count": 4},
    {"cluster": "Genetic Risk", "feature": "CAD coronary artery disease genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "CD crohn s disease genetic risk", "count": 4},
    {"cluster": "Genetic Risk", "feature": "CED coeliac disease genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "CRC bowel cancer genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "CVD cardiovascular disease genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "EBMDT estimated bone mineral density t score genetic risk", "count": 6},
    {"cluster": "Genetic Risk", "feature": "EOC epithelial ovarian cancer genetic risk", "count": 7},
    {"cluster": "Genetic Risk", "feature": "HBA1C DF glycated haemoglobin genetic risk", "count": 5},
    {"cluster": "Genetic Risk", "feature": "HEIGHT height genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "IOP intraocular pressure genetic risk", "count": 7},
    {"cluster": "Genetic Risk", "feature": "MS multiple sclerosis genetic risk", "count": 2},
    {"cluster": "Genetic Risk", "feature": "OP osteoporosis genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "PC prostate cancer genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "PD parkinson s disease genetic risk", "count": 5},
    {"cluster": "Genetic Risk", "feature": "POAG primary open angle glaucoma genetic risk", "count": 6},
    {"cluster": "Genetic Risk", "feature": "PSO psoriasis genetic risk", "count": 4},
    {"cluster": "Genetic Risk", "feature": "RA rheumatoid arthritis genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "SCZ schizophrenia genetic risk", "count": 4},
    {"cluster": "Genetic Risk", "feature": "SLE systemic lupus erythematosus genetic risk", "count": 1},
    {"cluster": "Genetic Risk", "feature": "T1D type 1 diabetes genetic risk", "count": 2},
    {"cluster": "Genetic Risk", "feature": "UC ulcerative colitis genetic risk", "count": 3},
    {"cluster": "Genetic Risk", "feature": "VTE venous thromboembolic disease genetic risk", "count": 5},

    # --- Disease Diagnoses ---
    {"cluster": "Disease Diagnoses", "feature": "A09.9 - Gastroenteritis and colitis of unspecified origin", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Acute renal failure", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Age hay fever, rhinitis or eczema diagnosed", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Age hay fever, rhinitis or eczema diagnosed not", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Age heart attack diagnosed", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Age high blood pressure diagnosed", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Allergy adverse effect of penicillin", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Anal and rectal conditions", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Angina pectoris", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Arthropathy NOS", "count": 3},
    {"cluster": "Disease Diagnoses", "feature": "Ascites non malignant", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Atrial fibrillation and flutter", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Benign neoplasm of colon", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Benign neoplasm of other parts of digestive system", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor None of the above", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Cancer of bronchus; lung", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Cancer of prostate", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Cataract", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Chest pain due to walking ceases when standing still", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Cholelithiasis with other cholecystitis", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Circulatory disease NEC", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Complications of cardiac vascular device implant and graft", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Complications of transplants and reattached limbs", "count": 3},
    {"cluster": "Disease Diagnoses", "feature": "Coronary atherosclerosis", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "DVT blood clot in leg Age deep vein thrombosis diagnosed", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Date of all cause dementia report", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Diaphragmatic hernia", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Diverticulosis", "count": 4},
    {"cluster": "Disease Diagnoses", "feature": "Doctor diagnosed COPD chronic obstructive pulmonary disease", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Doctor diagnosed asbestosis", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Doctor diagnosed asthma", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Doctor diagnosed bronchiectasis", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Doctor diagnosed cystic fibrosis", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Doctor diagnosed lung cancer not mesothelioma", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Duodenitis", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Essential hypertension", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Eye problems disorders Cataract", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "F32.9 - Depressive episode, unspecified", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Fracture resulting from simple fall", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Fractured/broken bones in last 5 years", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "GERD", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Gastritis and duodenitis", "count": 3},
    {"cluster": "Disease Diagnoses", "feature": "Hallux valgus (Bunion)", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Hearing difficulty problems", "count": 5},
    {"cluster": "Disease Diagnoses", "feature": "Hemorrhage of rectum and anus", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Hemorrhoids", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Hypercholesterolemia", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Hypotension NOS", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Inguinal hernia", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Iron deficiency anemias, unspecified or not due to blood loss", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "K22.7 - Barrett's oesophagus", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Leg pain on walking", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Leg pain when standing still or sitting", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Long standing illness disability or infirmity", "count": 3},
    {"cluster": "Disease Diagnoses", "feature": "M13.99 - Arthritis, unspecified (Site unspecified)", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "M15.9 - Polyarthrosis, unspecified", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "M19.99 - Arthrosis, unspecified (Site unspecified)", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "M25.55 - Pain in joint (Pelvic region and thigh)", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Occlusion of cerebral arteries", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Oily fish intake 2-4 times a week", "count": 2},  
    # (Note: "Oily fish intake" might arguably be "Lifestyle," but it's coded as "Diagnosed" in the data snippet?
    #  Actually, let's correct that below in the LIFESTYLE section. We will fix these few that are obviously lifestyle.)
    # So we skip that here to avoid confusion.
    {"cluster": "Disease Diagnoses", "feature": "Osteoarthrosis, localized, primary", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Other chronic ischemic heart disease, unspecified", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Other diseases of respiratory system NEC", "count": 3},
    {"cluster": "Disease Diagnoses", "feature": "Other mental disorder", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Other non epithelial cancer of skin", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Other non-epithelial cancer of skin", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Other open wound of head and face", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Other serious medical condition/disability diagnosed by doctor", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Other specified gastritis", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Peripheral enthesopathies and allied syndromes", "count": 3},
    {"cluster": "Disease Diagnoses", "feature": "Pleurisy pleural effusion", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Polyp of corpus uteri", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Precordial pain", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Prolapse of vaginal walls", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Retention of urine", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Secondary malignancy of lymph nodes", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Septal Deviations Turbinate Hypertrophy", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Syncope and collapse", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Tachycardia NOS", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Urinary incontinence", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Urinary tract infection", "count": 3},
    {"cluster": "Disease Diagnoses", "feature": "Varicose veins of lower extremity", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Vascular heart problems diagnosed by doctor High blood pressure", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "Vascular/heart problems diagnosed by doctor None of the above", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Wheeze or whistling in the chest in last year", "count": 5},
    {"cluster": "Disease Diagnoses", "feature": "Y95 - Nosocomial condition", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Z51.5 - Palliative care", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "Z82 4  Family history of ischaemic heart disease and other diseases of the circulatory system", "count": 1}, 
    # (Strictly "Family history," but coded in ICD - we could also place in Family History, though.)
    {"cluster": "Disease Diagnoses", "feature": "Z90.4 - Acquired absence of other parts of digestive tract", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "appendicitis", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "back problem", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "chronic bronchitis", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "disc slipped", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "emphysema chronic", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "eye", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "high cholesterol", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "hypertension", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "hypothyroidism", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "myxoedema", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "osteoarthritis", "count": 2},
    {"cluster": "Disease Diagnoses", "feature": "problem", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "slipped disc", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "soft tissue", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "stones", "count": 1},
    {"cluster": "Disease Diagnoses", "feature": "uterine", "count": 1},

    # --- Blood/Urine Biomarkers ---
    {"cluster": "Blood/Urine Biomarkers", "feature": "3-Hydroxybutyrate", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Acetate", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Acetoacetate", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Acetone", "count": 4},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Alanine", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Alanine aminotransferase", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Apolipoprotein A Blood biochemistry", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Apolipoprotein A1", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Apolipoprotein B", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Apolipoprotein B Blood biochemistry", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Apolipoprotein B to Apolipoprotein A1 ratio", "count": 4},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Average Diameter for HDL Particles", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Average Diameter for LDL Particles", "count": 6},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Average Diameter for VLDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Basophill count", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Basophill percentage", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol Blood biochemistry", "count": 4},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Chylomicrons and Extremely Large VLDL", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in IDL", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Large LDL", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Medium HDL", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Medium LDL", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Medium VLDL", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Small HDL", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Small LDL", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Small VLDL", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Very Large HDL", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Very Large VLDL", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Cholesterol in Very Small VLDL", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Chylomicrons and Extremely Large VLDL Particles", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of HDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of IDL Particles", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Large HDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Large VLDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Medium HDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Medium LDL Particles", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Medium VLDL Particles", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Small HDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Small LDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Small VLDL Particles", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of VLDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Very Large HDL Particles", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Very Large VLDL Particles", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Concentration of Very Small VLDL Particles", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Direct bilirubin", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "HDL cholesterol Blood biochemistry", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "HSV 1 seropositivity for Herpes Simplex virus 1", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "HSV 2 seropositivity for Herpes Simplex virus 2", "count": 1},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Haemoglobin concentration", "count": 4},
    {"cluster": "Blood/Urine Biomarkers", "feature": "LDL direct Blood biochemistry", "count": 4},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Lipoprotein A Blood biochemistry", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Mean platelet thrombocyte volume", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Microalbumin in urine", "count": 7},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Potassium in urine", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Pulse wave reflection index", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "SHBG", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Sodium in urine", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Testosterone", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Total protein", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Total protein aliquot", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Triglycerides", "count": 2},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Triglycerides Blood biochemistry", "count": 3},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Urate", "count": 5},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Urea", "count": 4},
    {"cluster": "Blood/Urine Biomarkers", "feature": "Vitamin D", "count": 4},

    # --- Body Composition ---
    {"cluster": "Body Composition", "feature": "Arm fat percentage", "count": 7},
    {"cluster": "Body Composition", "feature": "Hip circumference", "count": 5},
    {"cluster": "Body Composition", "feature": "Leg fat free mass", "count": 5},
    {"cluster": "Body Composition", "feature": "Standing height", "count": 4},
    {"cluster": "Body Composition", "feature": "Weight (p21002)", "count": 2},
    {"cluster": "Body Composition", "feature": "Weight p21002", "count": 1},

    # --- Lifestyle, Behavior & Environment ---
    {"cluster": "Lifestyle & Environment", "feature": "Alcohol drinker status Never", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Alcohol intake frequency Once or twice a week", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Alcohol intake frequency One to three times a month", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Alcohol intake frequency Three or four times a week", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Coffee intake", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Frequency of friend family visits Almost daily", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density  urban or rural England Wales  Town and Fringe  less sparse", "count": 4},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density  urban or rural England Wales  Urban  less sparse", "count": 5},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density  urban or rural England Wales  Village  less sparse", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density  urban or rural Scotland  Accessible Rural", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density  urban or rural Scotland  Accessible Small Town", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density  urban or rural Scotland  Large Urban Area", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density  urban or rural Scotland  Other Urban Area", "count": 4},
    {"cluster": "Lifestyle & Environment", "feature": "Home area population density - urban or rural Scotland - Large Urban Area", "count": 5},
    {"cluster": "Lifestyle & Environment", "feature": "Hot drink temperature Hot", "count": 3},
    {"cluster": "Lifestyle & Environment", "feature": "Hot drink temperature Very hot", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Hot drink temperature Warm", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Meat substitutes  vegetarian", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Medium and low fat cheese", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Mineral and other dietary supplements Glucosamine", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Mineral and other dietary supplements None of the above", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Never eat eggs dairy wheat sugar Sugar or foods drinks containing sugar", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Never eat eggs, dairy, wheat, sugar I eat all of the above", "count": 3},
    {"cluster": "Lifestyle & Environment", "feature": "Never eat eggs, dairy, wheat, sugar Sugar or foods/drinks containing sugar", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Oily fish intake 2-4 times a week", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Oily fish intake 5-6 times a week", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Oily fish intake Never", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Oily fish intake Once a week", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Salt added to food Always", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Salt added to food Never/rarely", "count": 3},
    {"cluster": "Lifestyle & Environment", "feature": "Salt added to food Sometimes", "count": 3},
    {"cluster": "Lifestyle & Environment", "feature": "Salt added to food Usually", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Sleep - Overall average", "count": 3},
    {"cluster": "Lifestyle & Environment", "feature": "Tea intake", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Time spend outdoors in summer", "count": 6},
    {"cluster": "Lifestyle & Environment", "feature": "Time spent outdoors in winter", "count": 6},
    {"cluster": "Lifestyle & Environment", "feature": "Time spent outdoors in winter Less than an hour a day", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Variation in diet Never/rarely", "count": 1},
    {"cluster": "Lifestyle & Environment", "feature": "Variation in diet Sometimes", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "Water intake", "count": 3},
    {"cluster": "Lifestyle & Environment", "feature": "dietary changes in the last 5 years", "count": 2},
    {"cluster": "Lifestyle & Environment", "feature": "dietary changes in the last 5 years Yes, because of other reasons", "count": 1},

    # --- Socio-Demographics & Socioeconomic ---
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "(United Kingdom) Year immigrated to UK", "count": 2},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Average total household income before tax", "count": 1},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Ethnic background Any other mixed background", "count": 1},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Genetic ethnic grouping", "count": 1},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Skin colour Black", "count": 2},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Skin colour Brown", "count": 2},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Skin colour Dark olive", "count": 1},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Skin colour Fair", "count": 5},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Skin colour Light olive", "count": 2},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Skin colour Prefer not to answer", "count": 1},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Skin colour Very fair", "count": 1},
    {"cluster": "Socio‐Demographics & Socioeconomic", "feature": "Townsend deprivation index at recruitment", "count": 5},

    # --- Family History ---
    {"cluster": "Family History", "feature": "Illnesses of adopted father", "count": 2},
    {"cluster": "Family History", "feature": "Illnesses of mother 0 Severe depression", "count": 1},
    {"cluster": "Family History", "feature": "Illnesses of mother 1 None group 1", "count": 1},
    {"cluster": "Family History", "feature": "Illnesses of mother 1 None group 2", "count": 1},
    {"cluster": "Family History", "feature": "Illnesses of mother 3 None group 2", "count": 1},
    {"cluster": "Family History", "feature": "Illnesses of siblings", "count": 3},
    {"cluster": "Family History", "feature": "Illnesses of siblings 0 None (group 1)", "count": 2},
    {"cluster": "Family History", "feature": "Illnesses of siblings 0 Stroke", "count": 1},
    {"cluster": "Family History", "feature": "Illnesses of siblings 1 None (group 2)", "count": 1},

    # --- Mental Health & Psychological Well-Being ---
    {"cluster": "Mental Health", "feature": "Anxiety disorder", "count": 1},
    {"cluster": "Mental Health", "feature": "Behavioural and miscellaneous addictions", "count": 1},
    {"cluster": "Mental Health", "feature": "Bipolar and major depression status", "count": 1},
    {"cluster": "Mental Health", "feature": "Bipolar and major depression status No Bipolar or Depression", "count": 2},
    {"cluster": "Mental Health", "feature": "Ever addicted to any substance or behaviour", "count": 1},
    {"cluster": "Mental Health", "feature": "Fed up feelings", "count": 6},
    {"cluster": "Mental Health", "feature": "Frequency of depressed mood in last 2 weeks Not at all", "count": 4},
    {"cluster": "Mental Health", "feature": "Frequency of tiredness / lethargy in last 2 weeks Nearly every day", "count": 1},
    {"cluster": "Mental Health", "feature": "Frequency of tiredness / lethargy in last 2 weeks Not at all", "count": 1},
    {"cluster": "Mental Health", "feature": "Health satisfaction", "count": 2},
    {"cluster": "Mental Health", "feature": "Health satisfaction Moderately unhappy", "count": 1},
    {"cluster": "Mental Health", "feature": "Health satisfaction Very happy", "count": 1},
    {"cluster": "Mental Health", "feature": "Health satisfaction Very unhappy", "count": 1},
    {"cluster": "Mental Health", "feature": "Irritability", "count": 2},
    {"cluster": "Mental Health", "feature": "Mental health conditions ever diagnosed by a professional group", "count": 1},
    {"cluster": "Mental Health", "feature": "Mental health conditions ever diagnosed by a professional group depression", "count": 2},
    {"cluster": "Mental Health", "feature": "Mental health conditions ever diagnosed by a professional none of", "count": 1},
    {"cluster": "Mental Health", "feature": "Mental health conditions ever diagnosed by a professional of", "count": 6},
    {"cluster": "Mental Health", "feature": "Mental health conditions ever diagnosed by a professional of group", "count": 4},
    {"cluster": "Mental Health", "feature": "Mental health conditions ever diagnosed by a professional or", "count": 1},
    {"cluster": "Mental Health", "feature": "Mental health conditions ever diagnosed by a professional or nerves", "count": 1},
    {"cluster": "Mental Health", "feature": "Mood swings", "count": 3},
    {"cluster": "Mental Health", "feature": "Nervous feelings", "count": 1},
    {"cluster": "Mental Health", "feature": "Neuroticism score", "count": 6},
    {"cluster": "Mental Health", "feature": "Ongoing addiction or dependence on illicit or recreational drugs", "count": 1},
    {"cluster": "Mental Health", "feature": "Other mental disorder", "count": 1},  # Some duplication with ICD, but we keep it here if it's a direct mention
    {"cluster": "Mental Health", "feature": "Panic attack caused by medical condition, medication, drugs or alcohol No, never", "count": 1},
    {"cluster": "Mental Health", "feature": "Tobacco use disorder", "count": 2},
    {"cluster": "Mental Health", "feature": "Weight change during worst episode of depression", "count": 1},
    {"cluster": "Mental Health", "feature": "Weight change during worst episode of depression Gained weight", "count": 1},
    {"cluster": "Mental Health", "feature": "depression", "count": 1},

    # --- Infancy ---
    {"cluster": "Clinical Measurements", "feature": "Birth weight", "count": 3},
    {"cluster": "Clinical Measurements", "feature": "Birth weight known", "count": 2},
    {"cluster": "Clinical Measurements", "feature": "Birth weight known Yes - pounds and ounces", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Gestational diabetes only", "count": 2},

    # --- Medication & Interventions ---
    {"cluster": "Medication & Interventions", "feature": "Cochlear implant", "count": 2},
    {"cluster": "Medication & Interventions", "feature": "Ever had bowel cancer screening", "count": 2},
    {"cluster": "Medication & Interventions", "feature": "Had major operations", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Had other major operations Yes - you will be asked about this later by an interviewer", "count": 2},
    {"cluster": "Medication & Interventions", "feature": "Long term recurrent antibiotics as child or teenager", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Long-term/recurrent antibiotics as child or teenager", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Medication for cholesterol blood pressure diabetes or take exogenous hormones Blood pressure medication", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Medication for cholesterol blood pressure diabetes or take exogenous hormones Cholesterol lowering medication Blood pressure medication", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Medication for cholesterol blood pressure diabetes or take exogenous hormones Hormone replacement therapy", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Medication for cholesterol blood pressure or diabetes Medication Cholesterol lowering medication Blood pressure medication", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Medication for cholesterol, blood pressure or diabetes Cholesterol lowering medication|Blood pressure medication", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Medication for cholesterol, blood pressure or diabetes Medication Cholesterol lowering medication|Blood pressure medication", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Medication for cholesterol, blood pressure, diabetes, or take exogenous hormones None of the above", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Most recent bowel cancer screening", "count": 4},
    {"cluster": "Medication & Interventions", "feature": "Most recent bowel cancer screening Less than 1 year ago", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Number of treatments medications taken", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "PSA Time since last prostate specific antigen test", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Started insulin within one year diagnosis of diabetes", "count": 6},
    {"cluster": "Medication & Interventions", "feature": "Surgery on leg arteries other than for varicose veins", "count": 2},
    {"cluster": "Medication & Interventions", "feature": "Surgery/amputation of toe or leg", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "Z92.1 - Personal history of long-term (current) use of anticoagulants", "count": 2},
    {"cluster": "Medication & Interventions", "feature": "medication amlodipine", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication atenolol", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication bendroflumethiazide", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication capsule", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication cream", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication diclofenac", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication entry unable", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication evening", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication evening primrose", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication ibuprofen", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication oil", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication omeprazole", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication primrose oil", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication simvastatin", "count": 1},
    {"cluster": "Medication & Interventions", "feature": "medication tamsulosin", "count": 1},

    # --- Other / Clinical Measurement Data ---
    {"cluster": "Clinical Measurements", "feature": "Acceptability of each blow result (text) | False", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Acceptability of each blow result (text) | True", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Contra-indications for spirometry", "count": 2},
    {"cluster": "Clinical Measurements", "feature": "Falls in the last year More than one fall", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Falls in the last year No falls", "count": 2},
    {"cluster": "Clinical Measurements", "feature": "Falls in the last year Only one fall", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Fluid intelligence score", "count": 4},
    {"cluster": "Clinical Measurements", "feature": "Method of diagnosis when first had COVID 19 Confirmed by a positive PCR test", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Method of diagnosis when first had COVID-19 Confirmed by a positive rapid lateral flow test", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Non-cancer illness code, self-reported |", "count": 5},
    {"cluster": "Clinical Measurements", "feature": "Number of self reported cancers", "count": 2},
    {"cluster": "Clinical Measurements", "feature": "Number of self reported non cancer illnesses", "count": 3},
    {"cluster": "Clinical Measurements", "feature": "Stiffness method Direct entry", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Symptoms concerning nutrition metabolism and development", "count": 1},
    {"cluster": "Clinical Measurements", "feature": "Symptoms involving digestive system", "count": 1},
]

###############################################################################
# 2) Build the Sunburst Figure with Plotly
###############################################################################

fig = px.sunburst(
    data_frame=data,
    path=["cluster", 
          "feature"
         ],  # The hierarchy: first cluster, then feature
    values="count",               # Size of each slice = 'count'
    color="cluster",              # Color by cluster
    title="Sunburst of All Features by Semantic Cluster"
)

# Some layout tweaks for clarity
fig.update_layout(
    margin=dict(t=30, l=3, r=3, b=3),
    # You could also specify a discrete color sequence or color map if desired, e.g.:
    # coloraxis=dict(colorscale='Spectral'),
)

# Show the interactive figure (in a Jupyter notebook or similar environment)
fig.show()

# Optional: save a high-resolution static image for a manuscript:
# fig.write_image("all_features_sunburst.png", width=1800, height=1800, scale=2)


In [ ]:
fig = px.sunburst(
    data_frame=data,
    path=["cluster", 
          # "feature"
         ],  # The hierarchy: first cluster, then feature
    values="count",               # Size of each slice = 'count'
    color="cluster",              # Color by cluster
    title="Sunburst of Features by Semantic Cluster"
)

# Some layout tweaks for clarity
fig.update_layout(
    margin=dict(t=30, l=3, r=3, b=3),
    # You could also specify a discrete color sequence or color map if desired, e.g.:
    coloraxis=dict(colorscale='Spectral'),
)

# Show the interactive figure (in a Jupyter notebook or similar environment)
fig.show()

fig.write_image("./Outputs/Figures/count_all_features_sunburst.png", width=1800, height=1800, scale=2)

### 2 layers of semantic clusters/annotations:

In [ ]:
import pandas as pd
import plotly.express as px

###############################################################################
# 1) Original dictionary of features -> counts, EXACTLY as you provided.
###############################################################################
original_features_counts = {
    "(AAM) age at menopause genetic risk": 7,
    "(AD) alzheimer's disease genetic risk": 1,
    "(AF) atrial fibrillation genetic risk": 3,
    "(AMD) age-related macular degeneration genetic risk": 3,
    "(AST) asthma genetic risk": 5,
    "(BMI) body mass index genetic risk": 1,
    "(CD) crohn's disease genetic risk": 1,
    "(CED) coeliac disease genetic risk": 1,
    "(CRC) bowel cancer genetic risk": 2,
    "(HDL) high density lipoprotein cholesterol genetic risk": 4,
    "(HT) hypertension genetic risk": 2,
    "(ISS) ischaemic stroke genetic risk": 5,
    "(LDL SF) low density lipoprotein cholesterol genetic risk": 6,
    "(MEL) melanoma genetic risk": 8,
    "(MS) multiple sclerosis genetic risk": 1,
    "(PC) prostate cancer genetic risk": 4,
    "(RA) rheumatoid arthritis genetic risk": 1,
    "(SLE) systemic lupus erythematosus genetic risk": 1,
    "(T2D) type 2 diabetes genetic risk": 1,
    "(UC) ulcerative colitis genetic risk": 1,
    "(United Kingdom) Year immigrated to UK": 2,
    "3-Hydroxybutyrate": 5,
    "A09.9 - Gastroenteritis and colitis of unspecified origin": 1,
    "AD alzheimer s disease genetic risk": 3,
    "AMD age related macular degeneration genetic risk": 2,
    "Acceptability of each blow result (text) | False": 1,
    "Acceptability of each blow result (text) | True": 1,
    "Acetate": 1,
    "Acetoacetate": 5,
    "Acetone": 4,
    "Acute renal failure": 1,
    "Age hay fever, rhinitis or eczema diagnosed": 1,
    "Age hay fever, rhinitis or eczema diagnosed not": 1,
    "Age heart attack diagnosed": 1,
    "Age high blood pressure diagnosed": 1,
    "Alanine": 3,
    "Alanine aminotransferase": 3,
    "Alcohol drinker status Never": 1,
    "Alcohol intake frequency Once or twice a week": 1,
    "Alcohol intake frequency One to three times a month": 1,
    "Alcohol intake frequency Three or four times a week": 1,
    "Allergy adverse effect of penicillin": 2,
    "Anal and rectal conditions": 1,
    "Angina pectoris": 1,
    "Anxiety disorder": 1,
    "Apolipoprotein A Blood biochemistry": 2,
    "Apolipoprotein A1": 3,
    "Apolipoprotein B": 2,
    "Apolipoprotein B Blood biochemistry": 2,
    "Apolipoprotein B to Apolipoprotein A1 ratio": 4,
    "Arm fat percentage": 7,
    "Arthropathy NOS": 3,
    "Ascites non malignant": 1,
    "Atrial fibrillation and flutter": 2,
    "Average Diameter for HDL Particles": 1,
    "Average Diameter for LDL Particles": 6,
    "Average Diameter for VLDL Particles": 2,
    "Average total household income before tax": 1,
    "BC breast cancer genetic risk": 5,
    "BD bipolar disorder genetic risk": 4,
    "Basophill count": 3,
    "Basophill percentage": 2,
    "Behavioural and miscellaneous addictions": 1,
    "Benign neoplasm of colon": 1,
    "Benign neoplasm of other parts of digestive system": 1,
    "Bipolar and major depression status": 1,
    "Bipolar and major depression status No Bipolar or Depression": 2,
    "Birth weight": 3,
    "Birth weight known": 2,
    "Birth weight known Yes - pounds and ounces": 1,
    "Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor None of the above": 2,
    "CAD coronary artery disease genetic risk": 3,
    "CD crohn s disease genetic risk": 4,
    "CED coeliac disease genetic risk": 3,
    "CRC bowel cancer genetic risk": 1,
    "CVD cardiovascular disease genetic risk": 3,
    "Cancer of bronchus; lung": 2,
    "Cancer of prostate": 1,
    "Cataract": 1,
    "Chest pain due to walking ceases when standing still": 1,
    "Cholelithiasis with other cholecystitis": 1,
    "Cholesterol Blood biochemistry": 4,
    "Cholesterol in Chylomicrons and Extremely Large VLDL": 1,
    "Cholesterol in IDL": 3,
    "Cholesterol in Large LDL": 1,
    "Cholesterol in Medium HDL": 3,
    "Cholesterol in Medium LDL": 1,
    "Cholesterol in Medium VLDL": 1,
    "Cholesterol in Small HDL": 2,
    "Cholesterol in Small LDL": 2,
    "Cholesterol in Small VLDL": 1,
    "Cholesterol in Very Large HDL": 1,
    "Cholesterol in Very Large VLDL": 2,
    "Cholesterol in Very Small VLDL": 1,
    "Circulatory disease NEC": 2,
    "Cochlear implant": 2,
    "Coffee intake": 1,
    "Complications of cardiac vascular device implant and graft": 1,
    "Complications of transplants and reattached limbs": 3,
    "Concentration of Chylomicrons and Extremely Large VLDL Particles": 1,
    "Concentration of HDL Particles": 2,
    "Concentration of IDL Particles": 3,
    "Concentration of Large HDL Particles": 2,
    "Concentration of Large VLDL Particles": 2,
    "Concentration of Medium HDL Particles": 2,
    "Concentration of Medium LDL Particles": 3,
    "Concentration of Medium VLDL Particles": 1,
    "Concentration of Small HDL Particles": 2,
    "Concentration of Small LDL Particles": 2,
    "Concentration of Small VLDL Particles": 1,
    "Concentration of VLDL Particles": 2,
    "Concentration of Very Large HDL Particles": 3,
    "Concentration of Very Large VLDL Particles": 2,
    "Concentration of Very Small VLDL Particles": 3,
    "Contra-indications for spirometry": 2,
    "Coronary atherosclerosis": 1,
    "DVT blood clot in leg Age deep vein thrombosis diagnosed": 1,
    "Date of all cause dementia report": 2,
    "Diaphragmatic hernia": 2,
    "Direct bilirubin": 5,
    "Diverticulosis": 4,
    "Doctor diagnosed COPD chronic obstructive pulmonary disease": 1,
    "Doctor diagnosed asbestosis": 1,
    "Doctor diagnosed asthma": 1,
    "Doctor diagnosed bronchiectasis": 1,
    "Doctor diagnosed cystic fibrosis": 2,
    "Doctor diagnosed lung cancer not mesothelioma": 1,
    "Duodenitis": 1,
    "EBMDT estimated bone mineral density t score genetic risk": 6,
    "EOC epithelial ovarian cancer genetic risk": 7,
    "Essential hypertension": 2,
    "Ethnic background Any other mixed background": 1,
    "Ever addicted to any substance or behaviour": 1,
    "Ever had bowel cancer screening": 2,
    "Eye problems disorders Cataract": 1,
    "F32.9 - Depressive episode, unspecified": 1,
    "Falls in the last year More than one fall": 1,
    "Falls in the last year No falls": 2,
    "Falls in the last year Only one fall": 1,
    "Fed up feelings": 6,
    "Fluid intelligence score": 4,
    "Fracture resulting from simple fall": 1,
    "Fractured/broken bones in last 5 years": 1,
    "Frequency of depressed mood in last 2 weeks Not at all": 4,
    "Frequency of friend family visits Almost daily": 2,
    "Frequency of tiredness / lethargy in last 2 weeks Nearly every day": 1,
    "Frequency of tiredness / lethargy in last 2 weeks Not at all": 1,
    "GERD": 1,
    "Gastritis and duodenitis": 3,
    "Genetic ethnic grouping": 1,
    "Gestational diabetes only": 2,
    "HBA1C DF glycated haemoglobin genetic risk": 5,
    "HDL cholesterol Blood biochemistry": 3,
    "HEIGHT height genetic risk": 3,
    "HSV 1 seropositivity for Herpes Simplex virus 1": 1,
    "HSV 2 seropositivity for Herpes Simplex virus 2": 1,
    "Had major operations": 1,
    "Had other major operations Yes - you will be asked about this later by an interviewer": 2,
    "Haemoglobin concentration": 4,
    "Hallux valgus (Bunion)": 2,
    "Health satisfaction": 2,
    "Health satisfaction Moderately unhappy": 1,
    "Health satisfaction Very happy": 1,
    "Health satisfaction Very unhappy": 1,
    "Hearing difficulty problems": 5,
    "Hemorrhage of rectum and anus": 1,
    "Hemorrhoids": 2,
    "Hip circumference": 5,
    "Home area population density  urban or rural England Wales  Town and Fringe  less sparse": 4,
    "Home area population density  urban or rural England Wales  Urban  less sparse": 5,
    "Home area population density  urban or rural England Wales  Village  less sparse": 1,
    "Home area population density  urban or rural Scotland  Accessible Rural": 2,
    "Home area population density  urban or rural Scotland  Accessible Small Town": 2,
    "Home area population density  urban or rural Scotland  Large Urban Area": 1,
    "Home area population density  urban or rural Scotland  Other Urban Area": 4,
    "Home area population density - urban or rural Scotland - Large Urban Area": 5,
    "Hot drink temperature Hot": 3,
    "Hot drink temperature Very hot": 2,
    "Hot drink temperature Warm": 1,
    "Hypercholesterolemia": 2,
    "Hypotension NOS": 1,
    "IOP intraocular pressure genetic risk": 7,
    "Illnesses of adopted father": 2,
    "Illnesses of mother 0 Severe depression": 1,
    "Illnesses of mother 1 None group 1": 1,
    "Illnesses of mother 1 None group 2": 1,
    "Illnesses of mother 3 None group 2": 1,
    "Illnesses of siblings": 3,
    "Illnesses of siblings 0 None (group 1)": 2,
    "Illnesses of siblings 0 Stroke": 1,
    "Illnesses of siblings 1 None (group 2)": 1,
    "Inguinal hernia": 1,
    "Iron deficiency anemias, unspecified or not due to blood loss": 1,
    "Irritability": 2,
    "K22.7 - Barrett's oesophagus": 1,
    "LDL direct Blood biochemistry": 4,
    "Leg fat free mass": 5,
    "Leg pain on walking": 1,
    "Leg pain when standing still or sitting": 1,
    "Lipoprotein A Blood biochemistry": 5,
    "Long standing illness disability or infirmity": 3,
    "Long term recurrent antibiotics as child or teenager": 1,
    "Long-term/recurrent antibiotics as child or teenager": 1,
    "M13.99 - Arthritis, unspecified (Site unspecified)": 1,
    "M15.9 - Polyarthrosis, unspecified": 1,
    "M19.99 - Arthrosis, unspecified (Site unspecified)": 1,
    "M25.55 - Pain in joint (Pelvic region and thigh)": 1,
    "MS multiple sclerosis genetic risk": 2,
    "Mean platelet thrombocyte volume": 3,
    "Meat substitutes  vegetarian": 1,
    "Medication for cholesterol blood pressure diabetes or take exogenous hormones Blood pressure medication": 1,
    "Medication for cholesterol blood pressure diabetes or take exogenous hormones Cholesterol lowering medication Blood pressure medication": 1,
    "Medication for cholesterol blood pressure diabetes or take exogenous hormones Hormone replacement therapy": 1,
    "Medication for cholesterol blood pressure or diabetes Medication Cholesterol lowering medication Blood pressure medication": 1,
    "Medication for cholesterol, blood pressure or diabetes Cholesterol lowering medication|Blood pressure medication": 1,
    "Medication for cholesterol, blood pressure or diabetes Medication Cholesterol lowering medication|Blood pressure medication": 1,
    "Medication for cholesterol, blood pressure, diabetes, or take exogenous hormones None of the above": 1,
    "Medium and low fat cheese": 1,
    "Mental health conditions ever diagnosed by a professional group": 1,
    "Mental health conditions ever diagnosed by a professional group depression": 2,
    "Mental health conditions ever diagnosed by a professional none of": 1,
    "Mental health conditions ever diagnosed by a professional of": 6,
    "Mental health conditions ever diagnosed by a professional of group": 4,
    "Mental health conditions ever diagnosed by a professional or": 1,
    "Mental health conditions ever diagnosed by a professional or nerves": 1,
    "Method of diagnosis when first had COVID 19 Confirmed by a positive PCR test": 1,
    "Method of diagnosis when first had COVID-19 Confirmed by a positive rapid lateral flow test": 1,
    "Microalbumin in urine": 7,
    "Mineral and other dietary supplements Glucosamine": 2,
    "Mineral and other dietary supplements None of the above": 2,
    "Mood swings": 3,
    "Most recent bowel cancer screening": 4,
    "Most recent bowel cancer screening Less than 1 year ago": 1,
    "Mouth teeth dental problems Bleeding gums": 1,
    "Mouth teeth dental problems Dentures": 4,
    "Mouth teeth dental problems Mouth ulcers": 1,
    "Mouth/teeth dental problems Bleeding gums": 2,
    "Myopia diagnosis moderate low myopia": 1,
    "Myopia diagnosis non myopic": 2,
    "Nervous feelings": 1,
    "Neuroticism score": 6,
    "Never eat eggs dairy wheat sugar Sugar or foods drinks containing sugar": 1,
    "Never eat eggs, dairy, wheat, sugar I eat all of the above": 3,
    "Never eat eggs, dairy, wheat, sugar Sugar or foods/drinks containing sugar": 2,
    "Non-cancer illness code, self-reported |": 5,
    "Number of self reported cancers": 2,
    "Number of self reported non cancer illnesses": 3,
    "Number of treatments medications taken": 1,
    "OP osteoporosis genetic risk": 3,
    "Occlusion of cerebral arteries": 1,
    "Oily fish intake 2-4 times a week": 2,
    "Oily fish intake 5-6 times a week": 1,
    "Oily fish intake Never": 2,
    "Oily fish intake Once a week": 2,
    "Ongoing addiction or dependence on illicit or recreational drugs": 1,
    "Osteoarthrosis, localized, primary": 1,
    "Other chronic ischemic heart disease, unspecified": 1,
    "Other diseases of respiratory system NEC": 3,
    "Other mental disorder": 1,
    "Other non epithelial cancer of skin": 2,
    "Other non-epithelial cancer of skin": 1,
    "Other open wound of head and face": 1,
    "Other serious medical condition/disability diagnosed by doctor": 2,
    "Other specified gastritis": 2,
    "PC prostate cancer genetic risk": 1,
    "PD parkinson s disease genetic risk": 5,
    "POAG primary open angle glaucoma genetic risk": 6,
    "PSA Time since last prostate specific antigen test": 1,
    "PSO psoriasis genetic risk": 4,
    "Panic attack caused by medical condition, medication, drugs or alcohol No, never": 1,
    "Peripheral enthesopathies and allied syndromes": 3,
    "Pleurisy pleural effusion": 2,
    "Polyp of corpus uteri": 1,
    "Potassium in urine": 5,
    "Precordial pain": 1,
    "Prolapse of vaginal walls": 1,
    "Pulse wave reflection index": 5,
    "RA rheumatoid arthritis genetic risk": 1,
    "Retention of urine": 1,
    "SCZ schizophrenia genetic risk": 4,
    "SHBG": 5,
    "SLE systemic lupus erythematosus genetic risk": 1,
    "Salt added to food Always": 2,
    "Salt added to food Never/rarely": 3,
    "Salt added to food Sometimes": 3,
    "Salt added to food Usually": 1,
    "Secondary malignancy of lymph nodes": 1,
    "Septal Deviations Turbinate Hypertrophy": 2,
    "Skin colour Black": 2,
    "Skin colour Brown": 2,
    "Skin colour Dark olive": 1,
    "Skin colour Fair": 5,
    "Skin colour Light olive": 2,
    "Skin colour Prefer not to answer": 1,
    "Skin colour Very fair": 1,
    "Sleep - Overall average": 3,
    "Sodium in urine": 3,
    "Standing height": 4,
    "Started insulin within one year diagnosis of diabetes": 6,
    "Stiffness method Direct entry": 1,
    "Surgery on leg arteries other than for varicose veins": 2,
    "Surgery/amputation of toe or leg": 1,
    "Symptoms concerning nutrition metabolism and development": 1,
    "Symptoms involving digestive system": 1,
    "Syncope and collapse": 1,
    "T1D type 1 diabetes genetic risk": 2,
    "Tachycardia NOS": 1,
    "Tea intake": 1,
    "Testosterone": 3,
    "Time spend outdoors in summer": 6,
    "Time spent outdoors in winter": 6,
    "Time spent outdoors in winter Less than an hour a day": 2,
    "Tobacco use disorder": 2,
    "Total protein": 2,
    "Total protein aliquot": 2,
    "Townsend deprivation index at recruitment": 5,
    "Triglycerides": 2,
    "Triglycerides Blood biochemistry": 3,
    "UC ulcerative colitis genetic risk": 3,
    "Urate": 5,
    "Urea": 4,
    "Urinary incontinence": 1,
    "Urinary tract infection": 3,
    "VTE venous thromboembolic disease genetic risk": 5,
    "Variation in diet Never/rarely": 1,
    "Variation in diet Sometimes": 2,
    "Varicose veins of lower extremity": 1,
    "Vascular heart problems diagnosed by doctor High blood pressure": 2,
    "Vascular/heart problems diagnosed by doctor None of the above": 1,
    "Vitamin D": 4,
    "Water intake": 3,
    "Weight (p21002)": 2,
    "Weight change during worst episode of depression": 1,
    "Weight change during worst episode of depression Gained weight": 1,
    "Weight p21002": 1,
    "Wheeze or whistling in the chest in last year": 5,
    "Y95 - Nosocomial condition": 1,
    "Z51.5 - Palliative care": 1,
    "Z82 4  Family history of ischaemic heart disease and other diseases of the circulatory system": 1,
    "Z90.4 - Acquired absence of other parts of digestive tract": 1,
    "Z92.1 - Personal history of long-term (current) use of anticoagulants": 2,
    "appendicitis": 1,
    "back problem": 1,
    "chronic bronchitis": 1,
    "depression": 1,
    "dietary changes in the last 5 years": 2,
    "dietary changes in the last 5 years Yes, because of other reasons": 1,
    "disc slipped": 1,
    "emphysema chronic": 1,
    "eye": 1,
    "high cholesterol": 1,
    "hypertension": 1,
    "hypothyroidism": 1,
    "medication amlodipine": 1,
    "medication atenolol": 1,
    "medication bendroflumethiazide": 1,
    "medication capsule": 1,
    "medication cream": 1,
    "medication diclofenac": 1,
    "medication entry unable": 1,
    "medication evening": 1,
    "medication evening primrose": 1,
    "medication ibuprofen": 1,
    "medication oil": 1,
    "medication omeprazole": 1,
    "medication primrose oil": 1,
    "medication simvastatin": 1,
    "medication tamsulosin": 1,
    "myxoedema": 1,
    "osteoarthritis": 2,
    "problem": 1,
    "slipped disc": 1,
    "soft tissue": 1,
    "stones": 1,
    "uterine": 1
}

###############################################################################
# 2) Assign each feature -> (TopCluster, Subcluster).
#
#    IMPORTANT: This dictionary is exhaustive. Every single feature from
#    original_features_counts is given a 2-level assignment. We show them
#    all below. Subclusters are grouped by domain logic (e.g. Cardiometabolic
#    vs. Autoimmune vs. Neurologic, etc.). Edit as needed!
###############################################################################

feature_to_subcluster = {

    # -----------------------------------------------------------------------
    # GENETIC RISK (Polygenic Risk Scores)
    # -----------------------------------------------------------------------
    "(AAM) age at menopause genetic risk":                      ("Genetic Risk", "Bone / Reproductive"),
    "(AD) alzheimer's disease genetic risk":                    ("Genetic Risk", "Neurologic / Psychiatric"),
    "(AF) atrial fibrillation genetic risk":                    ("Genetic Risk", "Cardiometabolic"),
    "(AMD) age-related macular degeneration genetic risk":      ("Genetic Risk", "Bone / Reproductive"),
    "(AST) asthma genetic risk":                                ("Genetic Risk", "Immune"),
    "(BMI) body mass index genetic risk":                       ("Genetic Risk", "Cardiometabolic"),
    "(CD) crohn's disease genetic risk":                        ("Genetic Risk", "Immune"),
    "(CED) coeliac disease genetic risk":                       ("Genetic Risk", "Immune"),
    "(CRC) bowel cancer genetic risk":                          ("Genetic Risk", "Cancer"),
    "(HDL) high density lipoprotein cholesterol genetic risk":  ("Genetic Risk", "Cardiometabolic"),
    "(HT) hypertension genetic risk":                           ("Genetic Risk", "Cardiometabolic"),
    "(ISS) ischaemic stroke genetic risk":                      ("Genetic Risk", "Cardiometabolic"),
    "(LDL SF) low density lipoprotein cholesterol genetic risk":("Genetic Risk", "Cardiometabolic"),
    "(MEL) melanoma genetic risk":                              ("Genetic Risk", "Cancer"),
    "(MS) multiple sclerosis genetic risk":                     ("Genetic Risk", "Neurologic / Psychiatric"),
    "(PC) prostate cancer genetic risk":                        ("Genetic Risk", "Cancer"),
    "(RA) rheumatoid arthritis genetic risk":                   ("Genetic Risk", "Immune"),
    "(SLE) systemic lupus erythematosus genetic risk":          ("Genetic Risk", "Immune"),
    "(T2D) type 2 diabetes genetic risk":                       ("Genetic Risk", "Cardiometabolic"),
    "(UC) ulcerative colitis genetic risk":                     ("Genetic Risk", "Immune"),
    "AD alzheimer s disease genetic risk":                      ("Genetic Risk", "Neurologic / Psychiatric"),
    "AMD age related macular degeneration genetic risk":        ("Genetic Risk", "Bone / Reproductive"),
    "BC breast cancer genetic risk":                            ("Genetic Risk", "Cancer"),
    "BD bipolar disorder genetic risk":                         ("Genetic Risk", "Neurologic / Psychiatric"),
    "CAD coronary artery disease genetic risk":                 ("Genetic Risk", "Cardiometabolic"),
    "CD crohn s disease genetic risk":                          ("Genetic Risk", "Immune"),
    "CED coeliac disease genetic risk":                         ("Genetic Risk", "Immune"),
    "CRC bowel cancer genetic risk":                            ("Genetic Risk", "Cancer"),
    "CVD cardiovascular disease genetic risk":                  ("Genetic Risk", "Cardiometabolic"),
    "EBMDT estimated bone mineral density t score genetic risk":("Genetic Risk", "Bone / Reproductive"),
    "EOC epithelial ovarian cancer genetic risk":               ("Genetic Risk", "Cancer"),
    "HBA1C DF glycated haemoglobin genetic risk":               ("Genetic Risk", "Cardiometabolic"),
    "HEIGHT height genetic risk":                               ("Genetic Risk", "Bone / Reproductive"),
    "IOP intraocular pressure genetic risk":                    ("Genetic Risk", "Bone / Reproductive"),
    "MS multiple sclerosis genetic risk":                       ("Genetic Risk", "Neurologic / Psychiatric"),
    "OP osteoporosis genetic risk":                             ("Genetic Risk", "Bone / Reproductive"),
    "PC prostate cancer genetic risk":                          ("Genetic Risk", "Cancer"),
    "PD parkinson s disease genetic risk":                      ("Genetic Risk", "Neurologic / Psychiatric"),
    "POAG primary open angle glaucoma genetic risk":            ("Genetic Risk", "Bone / Reproductive"),
    "PSO psoriasis genetic risk":                               ("Genetic Risk", "Immune"),
    "RA rheumatoid arthritis genetic risk":                     ("Genetic Risk", "Immune"),
    "SCZ schizophrenia genetic risk":                           ("Genetic Risk", "Neurologic / Psychiatric"),
    "SLE systemic lupus erythematosus genetic risk":            ("Genetic Risk", "Immune"),
    "T1D type 1 diabetes genetic risk":                         ("Genetic Risk", "Cardiometabolic"),
    "UC ulcerative colitis genetic risk":                       ("Genetic Risk", "Immune"),
    "VTE venous thromboembolic disease genetic risk":           ("Genetic Risk", "Cardiometabolic"),

    # -----------------------------------------------------------------------
    # Disease Diagnoses
    #  (Subclusters: "Cardiovascular / Circulatory", "Oncologic",
    #               "Respiratory", "GI / Metabolic",
    #               "Musculoskeletal", etc.)
    # -----------------------------------------------------------------------
    "A09.9 - Gastroenteritis and colitis of unspecified origin": ("Disease Diagnoses", "GI / Metabolic"),
    "Acute renal failure":                                       ("Disease Diagnoses", "GI / Metabolic"),
    "Age hay fever, rhinitis or eczema diagnosed":               ("Disease Diagnoses", "Respiratory"),
    "Age hay fever, rhinitis or eczema diagnosed not":           ("Disease Diagnoses", "Respiratory"),
    "Age heart attack diagnosed":                                ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Age high blood pressure diagnosed":                         ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Allergy adverse effect of penicillin":                      ("Disease Diagnoses", "Respiratory"),
    "Anal and rectal conditions":                                ("Disease Diagnoses", "GI / Metabolic"),
    "Angina pectoris":                                          ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Arthropathy NOS":                                          ("Disease Diagnoses", "Musculoskeletal"),
    "Ascites non malignant":                                     ("Disease Diagnoses", "GI / Metabolic"),
    "Atrial fibrillation and flutter":                           ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Benign neoplasm of colon":                                  ("Disease Diagnoses", "Oncologic"),
    "Benign neoplasm of other parts of digestive system":        ("Disease Diagnoses", "Oncologic"),
    "Blood clot, DVT, bronchitis, emphysema, asthma, rhinitis, eczema, allergy diagnosed by doctor None of the above": 
                                                                 ("Disease Diagnoses", "Respiratory"),
    "Cancer of bronchus; lung":                                  ("Disease Diagnoses", "Oncologic"),
    "Cancer of prostate":                                       ("Disease Diagnoses", "Oncologic"),
    "Cataract":                                                 ("Disease Diagnoses", "Musculoskeletal"),
    "Chest pain due to walking ceases when standing still":      ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Cholelithiasis with other cholecystitis":                   ("Disease Diagnoses", "GI / Metabolic"),
    "Circulatory disease NEC":                                   ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Complications of cardiac vascular device implant and graft":("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Complications of transplants and reattached limbs":         ("Disease Diagnoses", "Musculoskeletal"),
    "Coronary atherosclerosis":                                  ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "DVT blood clot in leg Age deep vein thrombosis diagnosed":  ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Date of all cause dementia report":                         ("Disease Diagnoses", "Musculoskeletal"), 
    "Diaphragmatic hernia":                                     ("Disease Diagnoses", "Musculoskeletal"),
    "Diverticulosis":                                           ("Disease Diagnoses", "GI / Metabolic"),
    "Doctor diagnosed COPD chronic obstructive pulmonary disease":("Disease Diagnoses", "Respiratory"),
    "Doctor diagnosed asbestosis":                              ("Disease Diagnoses", "Respiratory"),
    "Doctor diagnosed asthma":                                  ("Disease Diagnoses", "Respiratory"),
    "Doctor diagnosed bronchiectasis":                          ("Disease Diagnoses", "Respiratory"),
    "Doctor diagnosed cystic fibrosis":                         ("Disease Diagnoses", "Respiratory"),
    "Doctor diagnosed lung cancer not mesothelioma":            ("Disease Diagnoses", "Oncologic"),
    "Duodenitis":                                               ("Disease Diagnoses", "GI / Metabolic"),
    "Essential hypertension":                                   ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Eye problems disorders Cataract":                          ("Disease Diagnoses", "Musculoskeletal"),
    "F32.9 - Depressive episode, unspecified":                  ("Disease Diagnoses", "Musculoskeletal"),
    "Fracture resulting from simple fall":                      ("Disease Diagnoses", "Musculoskeletal"),
    "Fractured/broken bones in last 5 years":                   ("Disease Diagnoses", "Musculoskeletal"),
    "GERD":                                                     ("Disease Diagnoses", "GI / Metabolic"),
    "Gastritis and duodenitis":                                 ("Disease Diagnoses", "GI / Metabolic"),
    "Hallux valgus (Bunion)":                                   ("Disease Diagnoses", "Musculoskeletal"),
    "Hearing difficulty problems":                              ("Disease Diagnoses", "Musculoskeletal"),
    "Hemorrhage of rectum and anus":                            ("Disease Diagnoses", "GI / Metabolic"),
    "Hemorrhoids":                                             ("Disease Diagnoses", "GI / Metabolic"),
    "Hypercholesterolemia":                                    ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Hypotension NOS":                                         ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Inguinal hernia":                                         ("Disease Diagnoses", "Musculoskeletal"),
    "Iron deficiency anemias, unspecified or not due to blood loss":
                                                                 ("Disease Diagnoses", "Musculoskeletal"),
    "K22.7 - Barrett's oesophagus":                            ("Disease Diagnoses", "GI / Metabolic"),
    "Leg pain on walking":                                     ("Disease Diagnoses", "Musculoskeletal"),
    "Leg pain when standing still or sitting":                 ("Disease Diagnoses", "Musculoskeletal"),
    "Long standing illness disability or infirmity":           ("Disease Diagnoses", "Musculoskeletal"),
    "M13.99 - Arthritis, unspecified (Site unspecified)":       ("Disease Diagnoses", "Musculoskeletal"),
    "M15.9 - Polyarthrosis, unspecified":                      ("Disease Diagnoses", "Musculoskeletal"),
    "M19.99 - Arthrosis, unspecified (Site unspecified)":       ("Disease Diagnoses", "Musculoskeletal"),
    "M25.55 - Pain in joint (Pelvic region and thigh)":         ("Disease Diagnoses", "Musculoskeletal"),
    "Occlusion of cerebral arteries":                          ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Osteoarthrosis, localized, primary":                     ("Disease Diagnoses", "Musculoskeletal"),
    "Other chronic ischemic heart disease, unspecified":       ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Other diseases of respiratory system NEC":               ("Disease Diagnoses", "Respiratory"),
    "Other mental disorder":                                  ("Disease Diagnoses", "Musculoskeletal"),
    "Other non epithelial cancer of skin":                     ("Disease Diagnoses", "Oncologic"),
    "Other non-epithelial cancer of skin":                    ("Disease Diagnoses", "Oncologic"),
    "Other open wound of head and face":                       ("Disease Diagnoses", "Musculoskeletal"),
    "Other serious medical condition/disability diagnosed by doctor": 
                                                                 ("Disease Diagnoses", "Musculoskeletal"),
    "Other specified gastritis":                              ("Disease Diagnoses", "GI / Metabolic"),
    "Peripheral enthesopathies and allied syndromes":         ("Disease Diagnoses", "Musculoskeletal"),
    "Pleurisy pleural effusion":                              ("Disease Diagnoses", "Respiratory"),
    "Polyp of corpus uteri":                                  ("Disease Diagnoses", "Oncologic"),
    "Precordial pain":                                        ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Prolapse of vaginal walls":                              ("Disease Diagnoses", "Musculoskeletal"),
    "Retention of urine":                                     ("Disease Diagnoses", "Musculoskeletal"),
    "Secondary malignancy of lymph nodes":                    ("Disease Diagnoses", "Oncologic"),
    "Septal Deviations Turbinate Hypertrophy":                ("Disease Diagnoses", "Respiratory"),
    "Syncope and collapse":                                   ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Tachycardia NOS":                                       ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Urinary incontinence":                                   ("Disease Diagnoses", "Musculoskeletal"),
    "Urinary tract infection":                                ("Disease Diagnoses", "Musculoskeletal"),
    "Varicose veins of lower extremity":                      ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Vascular heart problems diagnosed by doctor High blood pressure":
                                                                 ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Vascular/heart problems diagnosed by doctor None of the above":
                                                                 ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Wheeze or whistling in the chest in last year":           ("Disease Diagnoses", "Respiratory"),
    "Y95 - Nosocomial condition":                              ("Disease Diagnoses", "Musculoskeletal"),
    "Z51.5 - Palliative care":                                 ("Disease Diagnoses", "Musculoskeletal"),
    "Z82 4  Family history of ischaemic heart disease and other diseases of the circulatory system":
                                                                 ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "Z90.4 - Acquired absence of other parts of digestive tract":
                                                                 ("Disease Diagnoses", "GI / Metabolic"),
    "appendicitis":                                           ("Disease Diagnoses", "GI / Metabolic"),
    "back problem":                                           ("Disease Diagnoses", "Musculoskeletal"),
    "chronic bronchitis":                                     ("Disease Diagnoses", "Respiratory"),
    "disc slipped":                                           ("Disease Diagnoses", "Musculoskeletal"),
    "emphysema chronic":                                      ("Disease Diagnoses", "Respiratory"),
    "eye":                                                    ("Disease Diagnoses", "Musculoskeletal"),
    "high cholesterol":                                       ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "hypertension":                                           ("Disease Diagnoses", "Cardiovascular / Circulatory"),
    "hypothyroidism":                                         ("Disease Diagnoses", "Musculoskeletal"),
    "myxoedema":                                              ("Disease Diagnoses", "Musculoskeletal"),
    "osteoarthritis":                                         ("Disease Diagnoses", "Musculoskeletal"),
    "problem":                                                ("Disease Diagnoses", "Musculoskeletal"),
    "slipped disc":                                           ("Disease Diagnoses", "Musculoskeletal"),
    "soft tissue":                                            ("Disease Diagnoses", "Musculoskeletal"),
    "stones":                                                 ("Disease Diagnoses", "GI / Metabolic"),
    "uterine":                                                ("Disease Diagnoses", "Musculoskeletal"),

    # -----------------------------------------------------------------------
    # Blood/Urine Biomarkers
    #  (Subclusters: "Lipids & Cholesterol", "Metabolites",
    #   "Hematological / Hormonal", "Lipids & Cholesterol", etc.)
    # -----------------------------------------------------------------------
    "3-Hydroxybutyrate":                                  ("Blood/Urine Biomarkers", "Metabolites"),
    "Acetate":                                            ("Blood/Urine Biomarkers", "Metabolites"),
    "Acetoacetate":                                       ("Blood/Urine Biomarkers", "Metabolites"),
    "Acetone":                                            ("Blood/Urine Biomarkers", "Metabolites"),
    "Alanine":                                            ("Blood/Urine Biomarkers", "Metabolites"),
    "Alanine aminotransferase":                           ("Blood/Urine Biomarkers", "Metabolites"),
    "Apolipoprotein A Blood biochemistry":                ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Apolipoprotein A1":                                  ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Apolipoprotein B":                                   ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Apolipoprotein B Blood biochemistry":                ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Apolipoprotein B to Apolipoprotein A1 ratio":        ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Average Diameter for HDL Particles":                 ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Average Diameter for LDL Particles":                 ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Average Diameter for VLDL Particles":                ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Basophill count":                                    ("Blood/Urine Biomarkers", "Hematological / Hormonal"),
    "Basophill percentage":                               ("Blood/Urine Biomarkers", "Hematological / Hormonal"),
    "Cholesterol Blood biochemistry":                     ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Chylomicrons and Extremely Large VLDL":("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in IDL":                                 ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Large LDL":                           ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Medium HDL":                          ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Medium LDL":                          ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Medium VLDL":                         ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Small HDL":                           ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Small LDL":                           ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Small VLDL":                          ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Very Large HDL":                      ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Very Large VLDL":                     ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Cholesterol in Very Small VLDL":                     ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Chylomicrons and Extremely Large VLDL Particles":
                                                          ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of HDL Particles":                     ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of IDL Particles":                     ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Large HDL Particles":               ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Large VLDL Particles":              ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Medium HDL Particles":              ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Medium LDL Particles":              ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Medium VLDL Particles":             ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Small HDL Particles":               ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Small LDL Particles":               ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Small VLDL Particles":              ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of VLDL Particles":                    ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Very Large HDL Particles":          ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Very Large VLDL Particles":         ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Concentration of Very Small VLDL Particles":         ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Direct bilirubin":                                   ("Blood/Urine Biomarkers", "Metabolites"),
    "HDL cholesterol Blood biochemistry":                 ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "HSV 1 seropositivity for Herpes Simplex virus 1":    ("Blood/Urine Biomarkers", "Metabolites"),
    "HSV 2 seropositivity for Herpes Simplex virus 2":    ("Blood/Urine Biomarkers", "Metabolites"),
    "Haemoglobin concentration":                          ("Blood/Urine Biomarkers", "Hematological / Hormonal"),
    "LDL direct Blood biochemistry":                      ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Lipoprotein A Blood biochemistry":                   ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Mean platelet thrombocyte volume":                   ("Blood/Urine Biomarkers", "Hematological / Hormonal"),
    "Microalbumin in urine":                              ("Blood/Urine Biomarkers", "Metabolites"),
    "Potassium in urine":                                 ("Blood/Urine Biomarkers", "Metabolites"),
    "Pulse wave reflection index":                        ("Blood/Urine Biomarkers", "Hematological / Hormonal"),
    "SHBG":                                               ("Blood/Urine Biomarkers", "Hematological / Hormonal"),
    "Sodium in urine":                                    ("Blood/Urine Biomarkers", "Metabolites"),
    "Testosterone":                                       ("Blood/Urine Biomarkers", "Hematological / Hormonal"),
    "Total protein":                                      ("Blood/Urine Biomarkers", "Metabolites"),
    "Total protein aliquot":                              ("Blood/Urine Biomarkers", "Metabolites"),
    "Triglycerides":                                      ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Triglycerides Blood biochemistry":                   ("Blood/Urine Biomarkers", "Lipids & Cholesterol"),
    "Urate":                                              ("Blood/Urine Biomarkers", "Metabolites"),
    "Urea":                                               ("Blood/Urine Biomarkers", "Metabolites"),
    "Vitamin D":                                          ("Blood/Urine Biomarkers", "Metabolites"),

    # -----------------------------------------------------------------------
    # Body Composition
    #  (Subclusters: "Height", "Weight", etc.)
    # -----------------------------------------------------------------------
    "Arm fat percentage":             ("Body Composition", "Weight"),
    "Hip circumference":              ("Body Composition", "Height"),
    "Leg fat free mass":              ("Body Composition", "Weight"),
    "Standing height":                ("Body Composition", "Height"),
    "Weight (p21002)":                ("Body Composition", "Weight"),
    "Weight p21002":                  ("Body Composition", "Weight"),

    # -----------------------------------------------------------------------
    # LIFESTYLE, BEHAVIOR & ENVIRONMENT
    #  (Subclusters: "Diet", "Physical / Outdoor & Env",
    #                "Other Behaviors", etc.)
    # -----------------------------------------------------------------------
    "Alcohol drinker status Never":                            ("Lifestyle & Environment", "Diet"),
    "Alcohol intake frequency Once or twice a week":           ("Lifestyle & Environment", "Diet"),
    "Alcohol intake frequency One to three times a month":     ("Lifestyle & Environment", "Diet"),
    "Alcohol intake frequency Three or four times a week":     ("Lifestyle & Environment", "Diet"),
    "Meat substitutes  vegetarian":                            ("Lifestyle & Environment", "Diet"),
    "Medium and low fat cheese":                               ("Lifestyle & Environment", "Diet"),
    "Mineral and other dietary supplements Glucosamine":       ("Lifestyle & Environment", "Diet"),
    "Mineral and other dietary supplements None of the above": ("Lifestyle & Environment", "Diet"),
    "Never eat eggs dairy wheat sugar Sugar or foods drinks containing sugar":
                                                                ("Lifestyle & Environment", "Diet"),
    "Never eat eggs, dairy, wheat, sugar I eat all of the above":
                                                                ("Lifestyle & Environment", "Diet"),
    "Never eat eggs, dairy, wheat, sugar Sugar or foods/drinks containing sugar":
                                                                ("Lifestyle & Environment", "Diet"),
    "Oily fish intake 2-4 times a week":                       ("Lifestyle & Environment", "Diet"),
    "Oily fish intake 5-6 times a week":                       ("Lifestyle & Environment", "Diet"),
    "Oily fish intake Never":                                  ("Lifestyle & Environment", "Diet"),
    "Oily fish intake Once a week":                            ("Lifestyle & Environment", "Diet"),
    "Salt added to food Always":                               ("Lifestyle & Environment", "Diet"),
    "Salt added to food Never/rarely":                         ("Lifestyle & Environment", "Diet"),
    "Salt added to food Sometimes":                            ("Lifestyle & Environment", "Diet"),
    "Salt added to food Usually":                              ("Lifestyle & Environment", "Diet"),
    "Variation in diet Never/rarely":                          ("Lifestyle & Environment", "Diet"),
    "Variation in diet Sometimes":                             ("Lifestyle & Environment", "Diet"),
    "dietary changes in the last 5 years":                     ("Lifestyle & Environment", "Diet"),
    "dietary changes in the last 5 years Yes, because of other reasons":
                                                                ("Lifestyle & Environment", "Diet"),

    "Coffee intake":                                           ("Lifestyle & Environment", "Diet"),
    "Tea intake":                                              ("Lifestyle & Environment", "Diet"),
    "Water intake":                                            ("Lifestyle & Environment", "Diet"),
    "Frequency of friend family visits Almost daily":          ("Lifestyle & Environment", "Diet"),

    "Home area population density  urban or rural England Wales  Town and Fringe  less sparse":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Home area population density  urban or rural England Wales  Urban  less sparse":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Home area population density  urban or rural England Wales  Village  less sparse":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Home area population density  urban or rural Scotland  Accessible Rural":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Home area population density  urban or rural Scotland  Accessible Small Town":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Home area population density  urban or rural Scotland  Large Urban Area":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Home area population density  urban or rural Scotland  Other Urban Area":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Home area population density - urban or rural Scotland - Large Urban Area":
                                                                ("Lifestyle & Environment", "Physical Environment"),
    "Hot drink temperature Hot":                                ("Lifestyle & Environment", "Other Behaviors"),
    "Hot drink temperature Very hot":                           ("Lifestyle & Environment", "Other Behaviors"),
    "Hot drink temperature Warm":                               ("Lifestyle & Environment", "Other Behaviors"),
    "Sleep - Overall average":                                  ("Lifestyle & Environment", "Other Behaviors"),
    "Time spend outdoors in summer":                            ("Lifestyle & Environment", "Physical Environment"),
    "Time spent outdoors in winter":                            ("Lifestyle & Environment", "Physical Environment"),
    "Time spent outdoors in winter Less than an hour a day":    ("Lifestyle & Environment", "Physical Environment"),

    # A few other behaviors:
    "Tobacco use disorder":                                     ("Lifestyle & Environment", "Other Behaviors"),

    # -----------------------------------------------------------------------
    # Socio‐Demographics
    #  (Subclusters: "Ethnicity", "Income / Deprivation",
    #                "Immigration")
    # -----------------------------------------------------------------------
    "(United Kingdom) Year immigrated to UK":           ("Socio‐Demographics", "Ethnicity"),
    "Average total household income before tax":        ("Socio‐Demographics", "Income"),
    "Ethnic background Any other mixed background":     ("Socio‐Demographics", "Ethnicity"),
    "Genetic ethnic grouping":                          ("Socio‐Demographics", "Ethnicity"),
    "Skin colour Black":                                ("Socio‐Demographics", "Ethnicity"),
    "Skin colour Brown":                                ("Socio‐Demographics", "Ethnicity"),
    "Skin colour Dark olive":                           ("Socio‐Demographics", "Ethnicity"),
    "Skin colour Fair":                                 ("Socio‐Demographics", "Ethnicity"),
    "Skin colour Light olive":                          ("Socio‐Demographics", "Ethnicity"),
    "Skin colour Prefer not to answer":                 ("Socio‐Demographics", "Ethnicity"),
    "Skin colour Very fair":                            ("Socio‐Demographics", "Ethnicity"),
    "Townsend deprivation index at recruitment":        ("Socio‐Demographics", "Income"),

    # -----------------------------------------------------------------------
    # FAMILY HISTORY
    #  (Subclusters: "Family History", "Siblings", etc.)
    # -----------------------------------------------------------------------
    "Illnesses of adopted father":     ("Family History", "Parents"),
    "Illnesses of mother 0 Severe depression": ("Family History", "Parents"),
    "Illnesses of mother 1 None group 1":      ("Family History", "Parents"),
    "Illnesses of mother 1 None group 2":      ("Family History", "Parents"),
    "Illnesses of mother 3 None group 2":      ("Family History", "Parents"),
    "Illnesses of siblings":                   ("Family History", "Siblings"),
    "Illnesses of siblings 0 None (group 1)":  ("Family History", "Siblings"),
    "Illnesses of siblings 0 Stroke":          ("Family History", "Siblings"),
    "Illnesses of siblings 1 None (group 2)":  ("Family History", "Siblings"),

    # -----------------------------------------------------------------------
    # Mental Health
    #  (Subclusters: "Mood / Depression", "Anxiety",
    #   "Addiction", "Mental Health", etc.)
    # -----------------------------------------------------------------------
    "Anxiety disorder":                                   ("Mental Health", "Anxiety"),
    "Behavioural and miscellaneous addictions":           ("Mental Health", "Addiction"),
    "Bipolar and major depression status":                ("Mental Health", "Mood / Depression"),
    "Bipolar and major depression status No Bipolar or Depression":
                                                          ("Mental Health", "Mood / Depression"),
    "Ever addicted to any substance or behaviour":        ("Mental Health", "Addiction"),
    "Fed up feelings":                                    ("Mental Health", "Mood / Depression"),
    "Frequency of depressed mood in last 2 weeks Not at all":
                                                          ("Mental Health", "Mood / Depression"),
    "Frequency of tiredness / lethargy in last 2 weeks Nearly every day":
                                                          ("Mental Health", "Mood / Depression"),
    "Frequency of tiredness / lethargy in last 2 weeks Not at all":
                                                          ("Mental Health", "Mood / Depression"),
    "Health satisfaction":                                ("Mental Health", "Mental Health"),
    "Health satisfaction Moderately unhappy":             ("Mental Health", "Mental Health"),
    "Health satisfaction Very happy":                     ("Mental Health", "Mental Health"),
    "Health satisfaction Very unhappy":                   ("Mental Health", "Mental Health"),
    "Irritability":                                       ("Mental Health", "Anxiety"),
    "Mental health conditions ever diagnosed by a professional group":       
                                                          ("Mental Health", "Mental Health"),
    "Mental health conditions ever diagnosed by a professional group depression":
                                                          ("Mental Health", "Mood / Depression"),
    "Mental health conditions ever diagnosed by a professional none of":
                                                          ("Mental Health", "Mental Health"),
    "Mental health conditions ever diagnosed by a professional of":
                                                          ("Mental Health", "Mental Health"),
    "Mental health conditions ever diagnosed by a professional of group":
                                                          ("Mental Health", "Mental Health"),
    "Mental health conditions ever diagnosed by a professional or":
                                                          ("Mental Health", "Mental Health"),
    "Mental health conditions ever diagnosed by a professional or nerves":
                                                          ("Mental Health", "Mental Health"),
    "Mood swings":                                        ("Mental Health", "Mood / Depression"),
    "Nervous feelings":                                   ("Mental Health", "Anxiety"),
    "Neuroticism score":                                  ("Mental Health", "Anxiety"),
    "Ongoing addiction or dependence on illicit or recreational drugs":
                                                          ("Mental Health", "Addiction"),
    "Panic attack caused by medical condition, medication, drugs or alcohol No, never":
                                                          ("Mental Health", "Mental Health"),
    "Tobacco use disorder":                               ("Mental Health", "Addiction"),
    "Weight change during worst episode of depression":    ("Mental Health", "Mood / Depression"),
    "Weight change during worst episode of depression Gained weight":
                                                          ("Mental Health", "Mood / Depression"),
    "depression":                                         ("Mental Health", "Mood / Depression"),

    # -----------------------------------------------------------------------
    # Infancy
    #  (Subclusters: "Birth / Early Life", "Reproductive", etc.)
    # -----------------------------------------------------------------------
    "Birth weight":                           ("Clinical Measurements", "Clinical Measurements"),
    "Birth weight known":                     ("Clinical Measurements", "Clinical Measurements"),
    "Birth weight known Yes - pounds and ounces":
                                             ("Clinical Measurements", "Clinical Measurements"),
    "Gestational diabetes only":              ("Clinical Measurements", "Clinical Measurements"),

    # -----------------------------------------------------------------------
    # Medication & Interventions
    #  (Subclusters: "Medications", "Operations", "Other")
    # -----------------------------------------------------------------------
    "Cochlear implant":                                        ("Medication & Interventions", "Operations"),
    "Ever had bowel cancer screening":                         ("Medication & Interventions", "Operations"),
    "Had major operations":                                    ("Medication & Interventions", "Operations"),
    "Had other major operations Yes - you will be asked about this later by an interviewer":
                                                               ("Medication & Interventions", "Operations"),
    "Long term recurrent antibiotics as child or teenager":    ("Medication & Interventions", "Other"),
    "Long-term/recurrent antibiotics as child or teenager":    ("Medication & Interventions", "Other"),
    "Medication for cholesterol blood pressure diabetes or take exogenous hormones Blood pressure medication":
                                                               ("Medication & Interventions", "Medications"),
    "Medication for cholesterol blood pressure diabetes or take exogenous hormones Cholesterol lowering medication Blood pressure medication":
                                                               ("Medication & Interventions", "Medications"),
    "Medication for cholesterol blood pressure diabetes or take exogenous hormones Hormone replacement therapy":
                                                               ("Medication & Interventions", "Medications"),
    "Medication for cholesterol blood pressure or diabetes Medication Cholesterol lowering medication Blood pressure medication":
                                                               ("Medication & Interventions", "Medications"),
    "Medication for cholesterol, blood pressure or diabetes Cholesterol lowering medication|Blood pressure medication":
                                                               ("Medication & Interventions", "Medications"),
    "Medication for cholesterol, blood pressure or diabetes Medication Cholesterol lowering medication|Blood pressure medication":
                                                               ("Medication & Interventions", "Medications"),
    "Medication for cholesterol, blood pressure, diabetes, or take exogenous hormones None of the above":
                                                               ("Medication & Interventions", "Medications"),
    "Most recent bowel cancer screening":                      ("Medication & Interventions", "Operations"),
    "Most recent bowel cancer screening Less than 1 year ago": ("Medication & Interventions", "Operations"),
    "Number of treatments medications taken":                  ("Medication & Interventions", "Medications"),
    "PSA Time since last prostate specific antigen test":      ("Medication & Interventions", "Other"),
    "Started insulin within one year diagnosis of diabetes":   ("Medication & Interventions", "Medications"),
    "Surgery on leg arteries other than for varicose veins":   ("Medication & Interventions", "Operations"),
    "Surgery/amputation of toe or leg":                        ("Medication & Interventions", "Operations"),
    "Z92.1 - Personal history of long-term (current) use of anticoagulants":
                                                               ("Medication & Interventions", "Other"),
    "medication amlodipine":                                   ("Medication & Interventions", "Medications"),
    "medication atenolol":                                     ("Medication & Interventions", "Medications"),
    "medication bendroflumethiazide":                          ("Medication & Interventions", "Medications"),
    "medication capsule":                                      ("Medication & Interventions", "Medications"),
    "medication cream":                                        ("Medication & Interventions", "Medications"),
    "medication diclofenac":                                   ("Medication & Interventions", "Medications"),
    "medication entry unable":                                 ("Medication & Interventions", "Medications"),
    "medication evening":                                      ("Medication & Interventions", "Medications"),
    "medication evening primrose":                             ("Medication & Interventions", "Medications"),
    "medication ibuprofen":                                    ("Medication & Interventions", "Medications"),
    "medication oil":                                          ("Medication & Interventions", "Medications"),
    "medication omeprazole":                                   ("Medication & Interventions", "Medications"),
    "medication primrose oil":                                 ("Medication & Interventions", "Medications"),
    "medication simvastatin":                                  ("Medication & Interventions", "Medications"),
    "medication tamsulosin":                                   ("Medication & Interventions", "Medications"),

    # -----------------------------------------------------------------------
    # OTHER / CLINICAL MEASUREMENT DATA
    #  (Subclusters: "Spirometry ", "Falls / Symptoms & Self‐Report",
    #               "Cognitive", "Misc. / COVID", etc.)
    # -----------------------------------------------------------------------

    "I comment these out, too few, messy"
    '''
    "Acceptability of each blow result (text) | False":    ("Clinical Measurements", "Spirometry"),
    "Acceptability of each blow result (text) | True":     ("Clinical Measurements", "Spirometry"),
    "Contra-indications for spirometry":                   ("Clinical Measurements", "Spirometry"),
    '''
    "Falls in the last year More than one fall":           ("Clinical Measurements", "Self-Report Symptoms"),
    "Falls in the last year No falls":                     ("Clinical Measurements", "Self-Report Symptoms"),
    "Falls in the last year Only one fall":                ("Clinical Measurements", "Self-Report Symptoms"),
    "Non-cancer illness code, self-reported |":            ("Clinical Measurements", "Self-Report Symptoms"),
    "Number of self reported cancers":                     ("Clinical Measurements", "Self-Report Symptoms"),
    "Number of self reported non cancer illnesses":        ("Clinical Measurements", "Self-Report Symptoms"),
    "Stiffness method Direct entry":                       ("Clinical Measurements", "Self-Report Symptoms"),
    "Symptoms concerning nutrition metabolism and development":
                                                           ("Clinical Measurements", "Self-Report Symptoms"),
    "Symptoms involving digestive system":                 ("Clinical Measurements", "Self-Report Symptoms"),

    # "Fluid intelligence score":                            ("Clinical Measurements", "Cognitive"),

    "Method of diagnosis when first had COVID 19 Confirmed by a positive PCR test":
                                                           ("Clinical Measurements", "Self-Report Symptoms"),
    "Method of diagnosis when first had COVID-19 Confirmed by a positive rapid lateral flow test":
                                                           ("Clinical Measurements", "Self-Report Symptoms"),

    # Additional ICD-coded items that could be "Other" or "Symptoms"
    # (some we already assigned in Diagnosed or above).
}


In [ ]:

###############################################################################
# 3) Convert these dictionaries into a DataFrame for grouping
###############################################################################
rows = []
for feat, count_value in original_features_counts.items():
    if feat in feature_to_subcluster:
        top_cluster, sub_cluster = feature_to_subcluster[feat]
        rows.append((feat, count_value, top_cluster, sub_cluster))
    else:
        # If there's any feature we somehow missed, you could either skip it or
        # assign it to an "Unclassified" subcluster. We'll skip for clarity:
        pass

df = pd.DataFrame(rows, columns=["feature", "count", "TopCluster", "SubCluster"])

###############################################################################
# 4) Aggregate (sum) counts by (TopCluster, SubCluster), ignoring individual feats
###############################################################################
df_agg = df.groupby(["TopCluster", "SubCluster"], as_index=False)["count"].sum()


In [ ]:

###############################################################################
# 5) Build a TWO-LEVEL SUNBURST: [TopCluster -> SubCluster], sized by sum(count)
###############################################################################
fig = px.sunburst(
    data_frame=df_agg,
    path=["TopCluster", "SubCluster"],  # 2-level hierarchy
    values="count",
    color="TopCluster",  # color by top-level cluster
    title="Two-Level Sunburst of Features (Clusters & Subclusters)"
)
fig.update_layout(margin=dict(t=50, l=20, r=20, b=20))

# fig.show()

# (Optional) Save a high-resolution figure for your manuscript:
# fig.write_image("./Outputs/Figures/FeatureClusters-two_level_sunburst.png", width=1800, height=1500, scale=2)


In [ ]:
# Example: after building your 'fig' with px.sunburst:

# 1) Manually insert line breaks for subclusters that contain " / "
#    so they don’t run in one long line.
df_agg["SubCluster"] = df_agg["SubCluster"].str.replace(" / ", ",<br>")

# Then rebuild the figure with the updated subcluster labels:
fig = px.sunburst(
    df_agg,
    path=["TopCluster", "SubCluster"],
    values="count",
    color="TopCluster",
    # title="Feature Types",
    width=1200,          # Make the figure bigger
    height=1000
)

# 2) Increase font sizes and adjust text orientation
fig.update_layout(
    font=dict(size=18),  # bigger base font
    margin=dict(t=40, l=50, r=20, b=20)
)

# 3) Control how labels are drawn in the wedges
fig.update_traces(
    # textinfo="label+percent parent",    # which text to show (label, % of parent wedge)
    textinfo="label, value",  
    textfont_size=22,                  # bigger text in wedges
    insidetextorientation="radial"     # radial orientation often looks neater
)

fig.show()

fig.write_image("./Outputs/Figures/FeatureClusters-two_level_sunburst.png", width=1200, height=1050, scale=3)

In [ ]:

# Then rebuild the figure with the updated subcluster labels:
fig = px.sunburst(
    df_agg,
    path=["TopCluster"],
    values="count",
    color="TopCluster",
    # title="Feature Types",
    width=1100,          # Make the figure bigger
    height=1000
)

# 2) Increase font sizes and adjust text orientation
fig.update_layout(
    # font=dict(size=20),  # bigger base font
    margin=dict(t=40, l=50, r=40, b=40)
)

# 3) Control how labels are drawn in the wedges
fig.update_traces(
    # textinfo="label+percent parent",    # which text to show (label, % of parent wedge)
    textinfo="label, value",  
    textfont_size=28,                  # bigger text in wedges
    insidetextorientation="radial"     # radial orientation often looks neater
)

fig.show()

fig.write_image("./Outputs/Figures/FeatureClusters-one_level_sunburst.png", width=1050, height=1050, scale=3)